In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()

In [0]:
def load_curated_customerrequestloan_cepalplatform(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args):
        date_filter = f"businessdate = {run_date!r}"

        cluster_columns = ["BusinessDate"]
        curated_table_path = f"{curated_zone}.Customer_Request_Loan"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "cepalplatform",
            )
        else:
            logging.info("Table does not exist, no need to delete entries for business date!")

        # ================== customerrequestloan ==================

        customerrequestloan_extra_filters = None

        if customerrequestloan_extra_filters is not None:
            customerrequestloan_extra_filters.append(date_filter)
            customerrequestloan_final_filters = " AND ".join(customerrequestloan_extra_filters)
        else:
            customerrequestloan_final_filters = date_filter

        customerrequestloan_selected_fields = [
            "CustomerRequestLoanID",
            "CustomerRequestID",
            "CustomerRequestVendorLoanID",
            "CustomerRequestLoanBalance",
            "CustomerRequestLoanIsSelected",
            "CustomerRequestLoanType",
            "CustomerRequestLoanSPVCategory",
            "BusinessDate",
        ]

        customerrequestloan = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cepalplatform_customerrequestloan",
            customerrequestloan_final_filters,
            customerrequestloan_selected_fields,
        )

        # ================== customerrequest (to get CDI) ==================

        customerrequest_extra_filters = None

        if customerrequest_extra_filters is not None:
            customerrequest_extra_filters.append(date_filter)
            customerrequest_final_filters = " AND ".join(customerrequest_extra_filters)
        else:
            customerrequest_final_filters = date_filter

        customerrequest_selected_fields = [
            "CustomerRequestID",
            "CustomerRequestCDI",
            "BusinessDate",
        ]

        customerrequest = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cepalplatform_customerrequest",
            customerrequest_final_filters,
            customerrequest_selected_fields,
        )

        # ========================== Joins ==========================

        customerrequestloan_customerrequest_join_condition = (
            (customerrequestloan.CustomerRequestID == customerrequest.CustomerRequestID)
            & (customerrequestloan.BusinessDate == customerrequest.BusinessDate)
        )

        target_table = customerrequestloan.join(
            customerrequest,
            customerrequestloan_customerrequest_join_condition,
            "left",
        )

        # ========================== Primary Key ==========================

        primary_key_fields = [
            customerrequestloan.CustomerRequestLoanID,
            customerrequestloan.CustomerRequestID,
            customerrequestloan.CustomerRequestVendorLoanID,
            lit("cepalplatform"),
        ]

        target_table = (
            target_table
            .withColumn(
                "CustomerRequestLoanSurrogateId",
                sha2(concat_ws(";", *primary_key_fields), 256),
            )
            .withColumn(
                "CustomerRequestLoanKey",
                concat_ws(";", customerrequestloan.CustomerRequestLoanID, customerrequestloan.CustomerRequestID, customerrequestloan.CustomerRequestVendorLoanID),
            )
        )

        # ========================== Select Final Columns ==========================

        result_df = target_table.select(
            col("CustomerRequestLoanSurrogateId").cast("string").alias("CustomerRequestLoanId"),
            target_table.CustomerRequestLoanKey.cast("string").alias("CustomerRequestLoanKey"),
            lit("cepalplatform").cast("string").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            customerrequestloan.CustomerRequestID.cast("int").alias("CustomerRequestKey"),
            customerrequest.CustomerRequestCDI.cast("string").alias("CustomerCDI"),
            customerrequestloan.CustomerRequestVendorLoanID.cast("string").alias("VendorLoanID"),
            customerrequestloan.CustomerRequestLoanBalance.cast("decimal(19,4)").alias("LoanBalance"),
            customerrequestloan.CustomerRequestLoanIsSelected.cast("boolean").alias("LoanIsSelected"),
            customerrequestloan.CustomerRequestLoanType.cast("string").alias("LoanType"),
            customerrequestloan.CustomerRequestLoanSPVCategory.cast("string").alias("LoanSPVCategory"),
            customerrequestloan.BusinessDate.alias("BusinessDate"),
        )

        # ========================== Append =========================

        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_customerrequestloan_cepalplatform(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)